In [72]:
import networkx as nx 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
from itertools import combinations 
from collections import Counter

Author: Isaiah Coriolan

In [110]:
# Load Network 

dist_edges_df = pd.read_csv("ATL_Transit_Edges.csv")
G_dir = nx.from_pandas_edgelist(dist_edges_df, create_using=nx.DiGraph(), source="source", target="target", edge_attr="weight")

Number of Nodes \& Edges

In [111]:
print(len(G_dir.nodes()))
print(len(G_dir.edges()))


8144
11788


Can we identify stops that have only incoming or outgoing connections?

In [4]:
# Nodes with no incoming edges
no_incoming = [n for n in G_dir.nodes() if G_dir.in_degree(n) == 0]

# Nodes with no outgoing edges
no_outgoing = [n for n in G_dir.nodes() if G_dir.out_degree(n) == 0]

print(no_incoming)

print(no_outgoing)

['Frey Rd at KSU Central Parking Deck', 'Central Avenue Southwest at Mitchell Street Southwest', 'Autry Circle @ Avalon', 'Interstate Pkwy @ Thornton']
['Frey Rd at Campus Loop', 'Mitchell Street Southwest at Central Avenue Southwest', 'Campbellton @ Spring', 'Thornton N @ Amazon', 'Arbor Place Mall @ Belk IB', 'Price @ Spring']


Can we find strongly connected components in the network?

In [96]:
num_strong_components = nx.number_strongly_connected_components(G_dir)
print(f"# of Strong Components: {num_strong_components}")

# Get strongly connected components
scc = list(nx.strongly_connected_components(G_dir))

# Each element in `scc` is a set of nodes in one strongly connected component
for i, component in enumerate(scc):
    print(f"Component {i+1}: {component}")

# Find the largest component
sorted_scc = sorted(scc, key = lambda x: len(x))
sorted_scc.reverse()
top = sorted_scc[:1]

# print(top)

data = []

# Create a dataframe from all of the components
for i, component in enumerate(sorted_scc):
    for item in component:
        data.append((i, item))

top_df = pd.DataFrame(data, columns = ["Component", "Station"])

stops = pd.read_csv("ATL_Stops.csv", header = 0, usecols = ["stop_name", "carrier"])
stops = stops.drop_duplicates()

top_df = pd.merge(top_df, stops, left_on = "Station", right_on = "stop_name", how = "inner")
top_df.drop_duplicates()

# Summarize the carrier mix within each component
summary = top_df.groupby(["Component" , "carrier"]).agg(Count = ("carrier", "count")).reset_index()

print(summary.head(50))

# Sort by count in descending order
summary = summary.sort_values(by = "Count", ascending = False)

print(summary.head(50))



# of Strong Components: 50
Component 1: {'Frey Rd at Campus Loop'}
Component 2: {'Mitchell Street Southwest at Central Avenue Southwest'}
Component 3: {'Martin Luther King Jr. Federal Building'}
Component 4: {'Old Alabama Rd SW at James Rd', 'Burns Rd & Hillcrest Rd (across)', 'Live Oak Pkwy & Wells Fargo Bank', 'Club Dr & Carrington Ct Apts (Court Dr)', 'Sugarloaf Pkwy & MacLeod Indusrial Park', 'Heritage Ct at Windy Ridge Pkwy', 'Mableton Pkwy at Old Powder Springs Rd', 'Old Norcross Rd & Sugarloaf Pkwy', 'Six Flags Pkwy SW at Bishop Rd', 'Spalding Dr & Technology Pkwy', 'Mableton Pkwy at Factory Shoals Rd', 'Collins Hill Rd Collins Industrial Way I', 'Buford Hwy & West Mount Tabor', 'Steve Reynolds Blvd & Windward Lane IB', 'Austell Rd at Reed Dr', 'Powder Springs Rd at Natchez Trace', 'Collins Hill Rd & Park Access Dr OB', 'Buford Hwy at Korean Catholic Church IB', 'Lawrenceville Hwy & Bartow Jenkins Park', 'Holcomb Bridge Rd & The Centre Apts', 'Lawrenceville Hwy & Kenvilla Dr IB'

Do any of these connected components include stops from different carriers? Or shared stops from different carriers?

In [99]:
# Map all of the stops in each component to their resepective carrier 
stops = pd.read_csv("ATL_Stops.csv", header = 0)

# List to store all component id's & stop names
data = []
for i, component in enumerate(scc):
    for item in component:
        data.append((i, item))

# Create commponent df 
component_df = pd.DataFrame(data, columns = ["component_id", "stop"])

# Merge w/stops 
component_df = pd.merge(component_df, stops, left_on = "stop", right_on = "stop_name", how = "inner")
component_df = component_df.loc[:, ["component_id", "stop", "carrier"]]
component_df = component_df.drop_duplicates()

# Group by stop name & find all transit carriers associated with a stop 
summary = component_df.groupby("stop").agg(Count = ("stop", "count")).reset_index()

# Find multiple stops 
multi_stops = summary[summary["Count"] > 1]
multi_stops = list(multi_stops["stop"])

# Trace these back to the df
filtered_df = component_df[component_df["stop"].isin(multi_stops)]

# Num of Shared Stops 
print(len(filtered_df))
print(filtered_df.head(50))
filtered_df.to_csv("filtered_df.csv")

# Group by stop and get unique carriers
grouped = filtered_df.groupby('stop')['carrier'].unique()

# For each stop, get all unique pairs of carriers
pair_counts = Counter()
for carriers in grouped:
    if len(carriers) > 1:
        for pair in combinations(sorted(carriers), 2):
            pair_counts[pair] += 1

# Convert to DataFrame 
pair_df = pd.DataFrame(pair_counts.items(), columns=['carrier_pair', 'count'])

print(pair_df.head())


30
      component_id                                               stop  \
363              3                           Peachtree St at Baker St   
364              3                           Peachtree St at Baker St   
516              3  Washington Street at Martin Luther King Junior...   
517              3  Washington Street at Martin Luther King Junior...   
559              3      Civic Center Station (West Peachtreet Street)   
561              3      Civic Center Station (West Peachtreet Street)   
572              3                           Courtland St at Ellis St   
573              3                           Courtland St at Ellis St   
601              3                 Peachtree Center Ave at Auburn Ave   
602              3                 Peachtree Center Ave at Auburn Ave   
812              3                  John Portman Blvd at Courtland St   
813              3                  John Portman Blvd at Courtland St   
825              3       North Avenue Station (W

What is the average distance between the stops of different transit carriers that have connections?

In [ ]:
# Load in df containing distances (miles) between stops
distance_df = pd.read_csv("ATL_Transit_Edges.csv", header = 0)

# Map all of the stops in each component to their resepective carrier 
stops = pd.read_csv("ATL_Stops.csv", header = 0, usecols = ["stop_name", "carrier"])

distance_df = pd.merge(distance_df, stops, left_on = "source", right_on = "stop_name", how = "inner")
distance_df.rename(columns = {"carrier": "source_carrier"}, inplace = True)

distance_df = pd.merge(distance_df, stops, left_on = "target", right_on = "stop_name", how = "inner")
distance_df.rename(columns = {"carrier": "target_carrier"}, inplace = True)
distance_df.drop(columns = ["stop_name_x", "stop_name_y"], inplace = True)

# Filter dataframe for where source_carrier != target_carrier 
distance_df = distance_df[distance_df["source_carrier"] != distance_df["target_carrier"]]

# print(distance_df.head())

summary = distance_df.groupby(["source_carrier", "target_carrier"]).agg(Avg_Distance = ("weight", "mean"))

print(summary.head(50))


                               Avg_Distance
source_carrier target_carrier              
clinc          gwinnett            7.771029
               xpress              0.242867
gwinnett       clinc               5.894080
               xpress              0.268000
xpress         clinc               0.226294
               gwinnett            0.260000


What are the Hub Stations in this network?

In [64]:
# Find the hubs/high-degree nodes 

hubs = nx.degree(G_dir)

# Ascending order
sorted_hubs = sorted(hubs, key = lambda x: x[1])

# Get descending order 
sorted_hubs.reverse()

print(sorted_hubs)

[('FIVE POINTS STATION', 74), ('NORTH SPRINGS STATION', 55), ('WEST END STATION', 53), ('EAST POINT STATION', 52), ('OAKLAND CITY STATION', 52), ('DORAVILLE STATION', 51), ('MIDTOWN STATION', 50), ('ARTS CENTER STATION', 49), ('DUNWOODY STATION', 48), ('LENOX STATION', 46), ('CIVIC CENTER STATION', 46), ('MEDICAL CENTER STATION', 46), ('SANDY SPRINGS STATION', 46), ('PEACHTREE CENTER STATION', 46), ('NORTH AVENUE STATION', 45), ('LAKEWOOD-FT MCPHERSON STATION', 44), ('BUCKHEAD STATION', 44), ('CHAMBLEE STATION', 44), ('COLLEGE PARK STATION', 44), ('AIRPORT STATION', 44), ('LINDBERGH CENTER STATION', 44), ('BROOKHAVEN-OGLETHORPE STATION', 44), ('GARNETT STATION', 44), ('KENSINGTON STATION', 42), ('BANKHEAD STATION', 34), ('INMAN PARK-REYNOLDSTOWN STATION', 30), ('KING MEMORIAL STATION', 30), ('GWCC-CNN CENTER STATION', 30), ('ASHBY STATION', 30), ('VINE CITY STATION', 30), ('GEORGIA STATE STATION', 30), ('EDGEWOOD-CANDLER PARK STATION', 30), ('DECATUR STATION', 30), ('EAST LAKE STATION'